# PatchTST + RevIN + series decomposition (rebuilding the MQL5 article, faithful pass)

**Rebuilds the approach from
[mql5.com/en/articles/15198](https://www.mql5.com/en/articles/15198)** ("PatchTST for
24-Hour Price Forecasting in MetaTrader 5") on top of this branch's current best result,
`hf_patchtst_revin_no_volume`. Source under `mql5/` (`patchTST.py`, `model_training.py`,
`model_prediction.py`) -- used as a reference, not run directly.

**This is a corrected, more faithful pass** after a first attempt
(`hf_patchtst_revin_decomposed`) didn't reproduce the article's characteristics as closely
as expected. A careful line-by-line comparison against `mql5/patchTST.py` found RevIN's
own math was already correct (equivalent to the article's when `affine=False`, which both
use), but **three other pieces silently didn't match**:

1. **Head mixed channels; the article's doesn't.** The article's `Flatten_Head`
   (`individual=False`) flattens each channel's own `(patch_num, d_model)` representation
   and applies **one shared linear layer independently per channel** -- no cross-channel
   mixing at all (the paper's actual channel-independence design). The first pass reused
   this project's usual fused head (concatenate all channels, one joint projection),
   inherited from the rest of this notebook family rather than the article. **Fixed here**:
   `PatchTSTOHLCVRevIN`'s head now flattens `(num_patches, d_model)` per channel and
   applies a single `nn.Linear(d_model * NUM_PATCHES, HORIZON)` with **no channel mixing**,
   matching `Flatten_Head` exactly (including its flatten-then-linear-then-dropout order).
2. **Positional encoding type didn't match.** The article uses `pe='zeros', learn_pe=True`
   -- a *learned* positional embedding, initialized small and uniform. HF's
   `PatchTSTConfig` defaults to `positional_encoding_type="sincos"` (fixed sinusoidal, not
   learned) and the first pass never overrode it. **Fixed here**: set
   `positional_encoding_type="random"` -- HF's closest equivalent (also a *learned*
   embedding, initialized `torch.randn` rather than the article's small uniform range, but
   the same "learned, not fixed" character `learn_pe=True` calls for; HF doesn't expose a
   literal "zeros"-style init option).
3. **Feed-forward width didn't match.** Article: `d_ff=256`. HF's default `ffn_dim=512`,
   never overridden. **Fixed here**: `ffn_dim=D_FF=256`.

**Renamed** `EXPERIMENT["name"]` to `hf_patchtst_revin_decomposed_faithful` (not reusing
`hf_patchtst_revin_decomposed`) so the first pass's result isn't silently overwritten --
both stay comparable.

**Kept from `hf_patchtst_revin_no_volume`/the first decomposition pass, deliberately, for
comparability**: SPY hourly data, `context_length=MAX_CONTEXT=70`/`HORIZON=3` (the
article's own `seq_len=168`/`pred_len=24` would make results incomparable to every other
row in `experiments.md`), OHLC only (no volume), real HF `PatchTSTModel` backbone instead
of the article's hand-rolled transformer, this family's own training hyperparameters
(`lr=6e-4`/`batch_size=512`/20 epochs, not the article's `lr=0.001`/`batch_size=32`/100
epochs), `channel_attention` swept True/False, `RevIN(affine=False)` and loss computed in
real price space (both already matched the article in the first pass).

**Deferred, same as every other notebook in this family**: the long-only bracket-order
backtest, until a target/volume/loss/architecture combination is picked.


## Colab setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "patchtst_ohlcv_mse"
REPO_DIR = "/content/ECE1508_GenAI"   # absolute path -- see note below

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    # git -C targets REPO_DIR explicitly rather than `cd X && ...`, so this is correct
    # regardless of the kernel's current working directory when the cell reruns.
    #
    # fetch + checkout {BRANCH} (not just `pull`): REPO_DIR persists across notebook runs
    # within the same Colab VM session, so if an earlier run in this session cloned a
    # DIFFERENT branch (e.g. this VM was previously used for a notebook with an older
    # BRANCH value), a bare `pull` here would just pull more commits onto that stale
    # branch instead of switching -- and the push cell at the bottom would then fail with
    # "src refspec {BRANCH} does not match any", since no local branch by that name would
    # exist to push. Explicitly checking out BRANCH every time this cell runs avoids that.
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

# Absolute path, not "ECE1508_GenAI": os.path.isdir("ECE1508_GenAI") above is checked
# relative to the CURRENT working directory -- on a second run of this cell (after the
# %cd below already moved the kernel into /content/ECE1508_GenAI), that relative check
# looks for /content/ECE1508_GenAI/ECE1508_GenAI, finds nothing, and silently clones a
# second, nested copy of the repo inside the first one instead of pulling it.
%cd {REPO_DIR}


In [ ]:
# torch is preinstalled on Colab; transformers is the one addition vs. steven's own
# colab_train.ipynb (steven's hand-rolled model has no library dependency for this).
!pip install -q transformers mplfinance pyyaml


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Imports and config

`sys.path` is set up so `from src.data_pipeline import ...` resolves the same way steven's
own scripts import it (inserts `steven/` itself onto the path, not the repo root).

`EXPERIMENT` describes *this run* -- echoed into each output JSON's `"experiment"` block
and the thing to paste into `experiments.md`'s row when logging the result.


In [ ]:
import sys, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from transformers import PatchTSTConfig, PatchTSTModel

if os.path.basename(os.getcwd()) == 'steven':
    os.chdir('..')
sys.path.insert(0, 'steven')

from src.data_pipeline import build_dataset, WindowSampler, MAX_CONTEXT, HORIZON
# No import of weighted_mse_loss/unpack_y/reconstruct_volume -- volume is dropped (matching
# both the article and hf_patchtst_revin_no_volume), and the loss here is plain MSE in real
# price space (see intro markdown), not the anchored-return weighted loss.

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DATA_PATH = 'steven/data/spy_ohlcv_1h.parquet'
OHLC_COLS = ['open', 'high', 'low', 'close']
N_CHANNELS = len(OHLC_COLS)   # 4 -- volume dropped, see intro markdown
CLOSE_IDX = OHLC_COLS.index('close')

# -- Describes this run -- echoed into every output JSON and comparison_metrics/ filename.
# "name" -> comparison_metrics/{name}-channel_attention_{false,true}.json
EXPERIMENT = {
    "name": "hf_patchtst_revin_decomposed_faithful",
    "description": (
        "Corrected, more faithful rebuild of mql5.com/en/articles/15198 on top of "
        "hf_patchtst_revin_no_volume: series decomposition (trend/residual, two RevIN "
        "forecasters summed) PLUS three fixes found via a line-by-line source comparison "
        "against mql5/patchTST.py that the first pass (hf_patchtst_revin_decomposed) "
        "missed -- genuinely channel-independent (non-mixing) head matching Flatten_Head, "
        "learned (non-sincos) positional encoding, and ffn_dim=256 matching the article's "
        "d_ff. Same task (context=70/horizon=3, SPY OHLC, no volume) as this branch's "
        "current best result for direct comparability."
    ),
    "target": "raw_price_revin_decomposed",
    "include_volume": False,
    "loss_weights": {"note": "unweighted MSE in real price space (trend_pred + residual_pred vs. true OHLC), matching the article"},
}

# -- Both settings are trained in one pass -- see the training loop below.
CHANNEL_ATTENTION_SETTINGS = [False, True]

# -- Decomposition + RevIN settings, taken from the article's Config (mql5/model_training.py).
KERNEL_SIZE = 25       # moving-average window for trend extraction -- see intro markdown caveat
REVIN_AFFINE = False   # article's own choice -- no learnable scale/shift, plain mean/std norm

# -- Model hyperparameters -- matched to steven/configs/patchtst.yaml's model block, same
# as every other notebook in this family, for comparable model capacity, EXCEPT D_FF --
# set to the article's own d_ff=256 (HF's own default is 512, see intro markdown fix #3).
D_MODEL      = 64
NUM_HEADS    = 4
NUM_LAYERS   = 3
D_FF         = 256   # matches the article's Config.d_ff -- HF's PatchTSTConfig defaults to 512
DROPOUT      = 0.1
HEAD_DROPOUT = 0.0

# -- Patch structure -- one trading day, non-overlapping.
PATCH_LEN    = 7
PATCH_STRIDE = 7
NUM_PATCHES  = (MAX_CONTEXT - PATCH_LEN) // PATCH_STRIDE + 1   # 10 -- context=70 divides evenly, no padding needed

# -- Training hyperparameters -- same as every other notebook in this family (not the
# article's own lr=0.001/batch_size=32/100-epoch values -- see intro markdown).
LR                      = 6e-4
WEIGHT_DECAY            = 1e-4
BATCH_SIZE              = 512
MAX_EPOCHS              = 20
TRAIN_WINDOWS_PER_EPOCH = 20000
WINDOWS_PER_EVAL_SET    = 3000

plt.rcParams['figure.figsize'] = (14, 4)

# -- Seed control -- reset again per setting in the training loop below, so both
# channel_attention runs get identical weight init, not just identical data order.
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f'EXPERIMENT = {EXPERIMENT}')
print(f'SEED = {SEED}, CHANNEL_ATTENTION_SETTINGS = {CHANNEL_ATTENTION_SETTINGS}')


## Series decomposition + RevIN + model

**`MovingAvg`/`SeriesDecomp`**: ported directly from the article's `patchTST.py`
(`moving_avg`/`series_decomp`). `MovingAvg` replication-pads both ends of the context by
`(kernel_size-1)//2` bars, then runs a centered `AvgPool1d` -- this keeps the trend
component's length equal to the input's, so `residual = raw - trend` is a plain
elementwise subtraction, no cropping needed.

**`RevIN`**: same mean/std-per-window mechanism as every earlier RevIN notebook in this
family, `affine=False` by default here (see intro markdown) -- when disabled,
`normalize`/`denormalize` skip the learnable scale/shift step entirely.

**`PatchTSTOHLCVRevIN`**: RevIN + HF `PatchTSTModel` backbone + **a genuinely
channel-independent head, matching the article's `Flatten_Head` (`individual=False`)
exactly** -- flattens each channel's own `(num_patches, d_model)` representation into one
vector, then applies a **single shared linear layer independently per channel** (broadcasts
over the channel dimension, same weights every channel, zero cross-channel mixing) to
project to `HORIZON`. This replaces the fused/mixing head used throughout the rest of this
notebook family. Returns already-denormalized real price directly (matches the article's
per-backbone denorm, so `PatchTSTDecomposed` can just sum two real-valued outputs).

**`PatchTSTDecomposed`**: decomposes the raw input once, runs the trend component through
`model_trend` and the residual component through an independently-initialized `model_res`
(each with its own RevIN statistics), sums both real-valued outputs -- matches the
article's `Model.forward()` (`x = res + trend`) exactly.


In [ ]:
class MovingAvg(nn.Module):
    """Centered moving average for trend extraction -- ported from the article's
    patchTST.py (moving_avg). Replication-pads both ends so output length == input length."""
    def __init__(self, kernel_size: int):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (bs, seq_len, channels)
        pad = (self.kernel_size - 1) // 2
        front = x[:, 0:1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)
        return x


class SeriesDecomp(nn.Module):
    """Splits a series into trend (moving average) + residual -- ported from the article's
    patchTST.py (series_decomp)."""
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        trend = self.moving_avg(x)
        residual = x - trend
        return residual, trend


class RevIN(nn.Module):
    """Reversible Instance Normalization (Kim et al. 2021). affine=False by default here,
    matching the article's own config (see intro markdown) -- plain per-window mean/std
    normalization, no learnable scale/shift, when affine is off."""
    def __init__(self, num_channels: int, eps: float = 1e-5, affine: bool = REVIN_AFFINE):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(num_channels))
            self.affine_bias = nn.Parameter(torch.zeros(num_channels))
        self.mean = None
        self.std = None

    def fit(self, context: torch.Tensor):
        self.mean = context.mean(dim=1, keepdim=True).detach()
        self.std = (context.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()

    def normalize(self, x: torch.Tensor) -> torch.Tensor:
        x = (x - self.mean) / self.std
        if self.affine:
            x = x * self.affine_weight + self.affine_bias
        return x

    def denormalize(self, x: torch.Tensor) -> torch.Tensor:
        if self.affine:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps ** 2)
        x = x * self.std + self.mean
        return x


class PatchTSTOHLCVRevIN(nn.Module):
    def __init__(self, config: PatchTSTConfig):
        super().__init__()
        self.revin = RevIN(num_channels=N_CHANNELS)
        self.backbone = PatchTSTModel(config)
        # Channel-independent head, matching the article's Flatten_Head(individual=False)
        # exactly: flatten (num_patches, d_model) per channel into one vector, then ONE
        # SHARED linear (same weights every channel, no cross-channel mixing) projects to
        # HORIZON. nn.Linear broadcasts over all leading dims (batch, channel) automatically,
        # so this one layer is applied independently per channel -- not the fused/mixing
        # head used elsewhere in this notebook family.
        self.flatten = nn.Flatten(start_dim=-2)
        self.head = nn.Linear(config.d_model * NUM_PATCHES, HORIZON)
        self.dropout = nn.Dropout(config.head_dropout)

    def forward(self, past_values_raw: torch.Tensor) -> torch.Tensor:
        """past_values_raw: (bs, MAX_CONTEXT, 4) raw OHLC (trend or residual component).
        Returns predicted future OHLC ALREADY DENORMALIZED to real price/real-scale units
        -- matches the article's per-backbone denorm, so PatchTSTDecomposed below can just
        sum two real-valued outputs directly."""
        self.revin.fit(past_values_raw)
        past_values_norm = self.revin.normalize(past_values_raw)

        base_out = self.backbone(past_values=past_values_norm)
        # last_hidden_state: (bs, num_channels, num_patches, d_model)
        z = self.flatten(base_out.last_hidden_state)         # (bs, num_channels, num_patches*d_model)
        raw = self.head(z)                                    # (bs, num_channels, HORIZON) -- shared weights per channel
        raw = self.dropout(raw)                               # flatten -> linear -> dropout, matches Flatten_Head's order
        pred_norm = raw.permute(0, 2, 1)                       # (bs, HORIZON, num_channels)

        return self.revin.denormalize(pred_norm)


class PatchTSTDecomposed(nn.Module):
    def __init__(self, config: PatchTSTConfig, kernel_size: int):
        super().__init__()
        self.decomp = SeriesDecomp(kernel_size)
        self.model_trend = PatchTSTOHLCVRevIN(config)
        self.model_res = PatchTSTOHLCVRevIN(config)

    def forward(self, past_values_raw: torch.Tensor) -> torch.Tensor:
        residual_in, trend_in = self.decomp(past_values_raw)
        trend_pred = self.model_trend(trend_in)
        res_pred = self.model_res(residual_in)
        return trend_pred + res_pred   # both already real-valued -- see PatchTSTOHLCVRevIN


def build_model(channel_attention: bool) -> PatchTSTDecomposed:
    config = PatchTSTConfig(
        num_input_channels=N_CHANNELS,   # 4 (open, high, low, close) -- volume dropped
        context_length=MAX_CONTEXT,      # 70 (fixed, matches every other notebook in this family)
        prediction_length=HORIZON,       # 3
        patch_length=PATCH_LEN,
        patch_stride=PATCH_STRIDE,
        d_model=D_MODEL,
        num_attention_heads=NUM_HEADS,
        num_hidden_layers=NUM_LAYERS,
        ffn_dim=D_FF,   # article's d_ff=256, not HF's default 512 -- see intro markdown fix #3
        # transformers no longer has a single unified "dropout" field on PatchTSTConfig
        # (split into these four sub-layer dropouts, see modeling_patchtst.py) -- broadcasting
        # the same rate to all four keeps this equivalent to a single nn.Dropout(p=dropout).
        attention_dropout=DROPOUT,
        positional_dropout=DROPOUT,
        path_dropout=DROPOUT,
        ff_dropout=DROPOUT,
        head_dropout=HEAD_DROPOUT,
        channel_attention=channel_attention,
        # Article: pe='zeros', learn_pe=True -- a LEARNED positional embedding. HF doesn't
        # expose a literal "zeros"-init option; "random" is its closest equivalent (also
        # learned, just torch.randn-initialized rather than small-uniform) -- see intro
        # markdown fix #2. Default "sincos" (fixed, not learned) would NOT match the article.
        positional_encoding_type="random",
        scaling=None,   # RevIN is applied externally instead -- see markdown above
    )
    # Same config object passed to both sub-models (matches the article's Model.__init__,
    # which reuses one Config for both model_trend/model_res) -- each PatchTSTOHLCVRevIN(config)
    # call below still constructs its own PatchTSTModel with independently-initialized weights.
    return PatchTSTDecomposed(config, KERNEL_SIZE)


## Data

Reuses `build_dataset`/`WindowSampler` from `steven/src/data_pipeline.py` for the
chronological train/val/test split boundaries and valid-start-index logic, and builds
windows directly from raw OHLC price (`df[OHLC_COLS]`) -- no volume, no
`build_window()`/return-decomposition, same as `hf_patchtst_revin_no_volume`.

Built **once** here, shared across both `channel_attention` settings in the training loop
below, same as every other notebook in this family.


In [ ]:
df, bounds, stats = build_dataset(DATA_PATH)
ohlc = df[OHLC_COLS].to_numpy(dtype=np.float32)   # (N, 4) -- raw price, no normalization yet, no volume

train_sampler = WindowSampler(*bounds['train'])
val_sampler   = WindowSampler(*bounds['val'])
test_sampler  = WindowSampler(*bounds['test'])

print(f'Train rows: {bounds["train"][1] - bounds["train"][0]:,}')
print(f'Val rows  : {bounds["val"][1] - bounds["val"][0]:,}')
print(f'Test rows : {bounds["test"][1] - bounds["test"][0]:,}')


class FixedContextOHLCDataset(Dataset):
    """Every window uses the full MAX_CONTEXT (70 bars); target is the next HORIZON (3)
    bars, both as raw OHLC -- no padding handling needed since ctx_bars == MAX_CONTEXT
    exactly (same fixed-context simplification as every other notebook in this family)."""
    def __init__(self, ohlc: np.ndarray, start_indices: np.ndarray):
        self.ohlc = ohlc
        self.start_indices = start_indices

    def __len__(self):
        return len(self.start_indices)

    def __getitem__(self, i):
        s = int(self.start_indices[i])
        context = self.ohlc[s : s + MAX_CONTEXT]
        target  = self.ohlc[s + MAX_CONTEXT : s + MAX_CONTEXT + HORIZON]
        return {
            'past_values': torch.from_numpy(context),
            'future_values': torch.from_numpy(target),
        }


def collate(batch: list[dict]) -> dict:
    return {
        'past_values': torch.stack([b['past_values'] for b in batch]),
        'future_values': torch.stack([b['future_values'] for b in batch]),
    }


train_starts_full = train_sampler.valid_starts(MAX_CONTEXT)
val_starts_full    = val_sampler.valid_starts(MAX_CONTEXT)
test_starts_full   = test_sampler.valid_starts(MAX_CONTEXT)
print(f'Valid train starts (ctx=70): {len(train_starts_full):,}')
print(f'Valid val starts (ctx=70)  : {len(val_starts_full):,}')
print(f'Valid test starts (ctx=70) : {len(test_starts_full):,}')

val_rng = np.random.default_rng(SEED)
n_val = min(WINDOWS_PER_EVAL_SET, len(val_starts_full))
val_starts = val_rng.choice(val_starts_full, size=n_val, replace=False)
val_ds = FixedContextOHLCDataset(ohlc, val_starts)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

test_rng = np.random.default_rng(123)   # matches steven/src/evaluate.py's --seed default
n_test = min(3000, len(test_starts_full))
test_starts = test_rng.choice(test_starts_full, size=n_test, replace=False)
test_ds = FixedContextOHLCDataset(ohlc, test_starts)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, collate_fn=collate)


## Training + evaluation

Trains one model per entry in `CHANNEL_ATTENTION_SETTINGS`, evaluates each on the same
fixed test set, and writes one JSON per setting to `steven/comparison_metrics/`.

**Loss**: plain MSE directly on real OHLC price (`F.mse_loss(pred, future_values)`) -- no
normalization of the target needed, since `PatchTSTDecomposed.forward()` already returns
denormalized predictions (see model markdown above). Matches the article's own loss
exactly (`MSELoss(outputs, batch_y)` in `mql5/model_training.py`).

**Evaluation** reports OHLC MAE/RMSE (already real-valued, no denormalization step
needed), per-horizon-bar directional accuracy, and the coherence rate -- same methodology
as `hf_patchtst_revin_no_volume`, for direct comparison.

**Comparability note**: this model only supports `context_length=70` (steven's "long"
bucket), not his full variable-length curriculum -- so compare against steven's **long**
bucket numbers in `steven/outputs/metrics.json`, not his "overall" (mixed-length) numbers.


In [ ]:
def run_epoch(model, loader, optimizer, device, train: bool, loss_history=None, step_holder=None, log_every=10):
    model.train(mode=train)
    total, n = 0.0, 0
    for batch in loader:
        past_values = batch['past_values'].to(device)
        future_values = batch['future_values'].to(device)

        with torch.set_grad_enabled(train):
            pred_real = model(past_values)
            # Plain MSE in real price space -- future_values is raw OHLC, no normalize()
            # step needed here since pred_real is already denormalized (see model markdown).
            loss = F.mse_loss(pred_real, future_values)

        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if step_holder is not None:
                step_holder[0] += 1
                if loss_history is not None and (step_holder[0] == 1 or step_holder[0] % log_every == 0):
                    loss_history.append((step_holder[0], float(loss.item())))

        bs = past_values.shape[0]
        total += loss.item() * bs
        n += bs
    return {'loss': total / n}


def mae_rmse(pred: np.ndarray, true: np.ndarray) -> tuple[float, float]:
    diff = pred - true
    return float(np.mean(np.abs(diff))), float(np.sqrt(np.mean(diff ** 2)))


def evaluate(model) -> dict:
    model.eval()
    all_true, all_pred, all_close0 = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            pv = batch['past_values'].to(DEVICE)
            fv = batch['future_values'].to(DEVICE)
            pred_real = model(pv)   # already real OHLC -- no denormalize() call needed here
            all_true.append(fv.cpu().numpy())
            all_pred.append(pred_real.cpu().numpy())
            all_close0.append(pv[:, -1, CLOSE_IDX].cpu().numpy())   # last context bar's real close

    true_ohlc = np.concatenate(all_true)     # (n, 3, 4) real [open,high,low,close]
    pred_ohlc = np.concatenate(all_pred)     # (n, 3, 4) predicted
    close_0 = np.concatenate(all_close0)     # (n,)
    n = len(true_ohlc)

    ohlc_mae, ohlc_rmse = mae_rmse(pred_ohlc, true_ohlc)

    true_close_sign = np.sign(true_ohlc[:, :, CLOSE_IDX] - close_0[:, None])
    pred_close_sign = np.sign(pred_ohlc[:, :, CLOSE_IDX] - close_0[:, None])
    dir_acc = (true_close_sign == pred_close_sign).mean(axis=0)

    # Coherence check -- see markdown above.
    coherent_up = (pred_close_sign > 0).all(axis=1)
    coherent_down = (pred_close_sign < 0).all(axis=1)
    coherence_rate = float((coherent_up | coherent_down).mean())

    return {
        'n_windows': n,
        'ohlc_mae': ohlc_mae,
        'ohlc_rmse': ohlc_rmse,
        'directional_accuracy': dir_acc.tolist(),
        'coherence_rate': coherence_rate,
    }


os.makedirs('steven/comparison_metrics', exist_ok=True)
results = {}

for channel_attention in CHANNEL_ATTENTION_SETTINGS:
    setting_key = f'channel_attention_{str(channel_attention).lower()}'
    print(f'\n{"="*70}\ntraining {setting_key}\n{"="*70}')

    # Reset the seed per setting so both runs get identical weight init/dropout draws --
    # data order is already independent of global RNG state (np.random.default_rng(SEED+epoch)
    # below), but model init/dropout use torch's global RNG, which would otherwise have
    # already been advanced by whichever setting trained first.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model = build_model(channel_attention).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    loss_history = []
    step_holder = [0]
    best_val_loss = float('inf')

    for epoch in range(MAX_EPOCHS):
        train_rng = np.random.default_rng(SEED + epoch)
        n_train = min(TRAIN_WINDOWS_PER_EPOCH, len(train_starts_full))
        train_starts = train_rng.choice(train_starts_full, size=n_train, replace=False)
        train_ds = FixedContextOHLCDataset(ohlc, train_starts)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

        train_metrics = run_epoch(model, train_loader, optimizer, DEVICE, train=True,
                                   loss_history=loss_history, step_holder=step_holder)
        val_metrics = run_epoch(model, val_loader, optimizer, DEVICE, train=False)

        print(f'epoch {epoch+1:2d}/{MAX_EPOCHS}  '
              f'train_loss={train_metrics["loss"]:.6f}  val_loss={val_metrics["loss"]:.6f}')

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']

    print(f'done ({setting_key}). best val_loss={best_val_loss:.6f}')

    fig, ax = plt.subplots(figsize=(10, 4))
    steps, losses = zip(*loss_history)
    ax.plot(steps, losses)
    ax.set_xlabel('training step')
    ax.set_ylabel('MSE loss (real price space, trend + residual summed)')
    ax.set_title(f'Training loss curve -- PatchTST + RevIN + decomposition, {setting_key}')
    plt.tight_layout()
    plt.show()

    eval_metrics = evaluate(model)
    print(f'  OHLC MAE / RMSE     : {eval_metrics["ohlc_mae"]:.4f} / {eval_metrics["ohlc_rmse"]:.4f}')
    dir_acc = eval_metrics['directional_accuracy']
    print(f'  dir acc per bar     : {dir_acc[0]:.4f} / {dir_acc[1]:.4f} / {dir_acc[2]:.4f}')
    print(f'  coherence rate      : {eval_metrics["coherence_rate"]:.4f}')
    print(f'\n  === For comparison: hf_patchtst_revin_no_volume (no decomposition), {setting_key} ===')
    if channel_attention:
        print('  OHLC MAE/RMSE 2.15/3.30, dir acc 0.529/0.544/0.549, coherence 0.987')
    else:
        print('  OHLC MAE/RMSE 1.92/3.11, dir acc 0.525/0.543/0.543, coherence 0.953')

    output = {
        'experiment': {**EXPERIMENT, 'channel_attention': channel_attention, 'seed': SEED},
        'best_val_loss': best_val_loss,
        **eval_metrics,
    }
    results[setting_key] = output

    out_path = f'steven/comparison_metrics/{EXPERIMENT["name"]}-{setting_key}.json'
    with open(out_path, 'w') as f:
        json.dump(output, f, indent=2)
    print(f'  wrote {out_path}')


## Results summary

In [ ]:
print(f'{"setting":<26} {"OHLC RMSE":>10} {"dir acc (bar1/2/3)":>22} {"coherence":>10}')
for setting_key, r in results.items():
    da = r['directional_accuracy']
    print(f'{setting_key:<26} {r["ohlc_rmse"]:>10.4f} '
          f'{da[0]:>6.4f}/{da[1]:.4f}/{da[2]:.4f}  {r["coherence_rate"]:>9.4f}')

print(f'\n  === For comparison: hf_patchtst_revin_no_volume (no decomposition) ===')
print('  channel_attention=False: OHLC RMSE 3.11, dir acc 0.525/0.543/0.543, coherence 0.953')
print('  channel_attention=True : OHLC RMSE 3.30, dir acc 0.529/0.544/0.549, coherence 0.987')


## Caveats and recommended next steps

- **Remaining approximations, not fully faithful even after this pass**:
  - HF's `positional_encoding_type="random"` uses `torch.randn` init; the article's
    `pe='zeros'` uses `nn.init.uniform_(-0.02, 0.02)`. Both are *learned* embeddings (the
    property that actually matters, `learn_pe=True`), but the init distribution differs.
  - `head_dropout=0.0` here (this family's own established default) vs. the article's
    `head_dropout=0.1` -- not changed, since it wasn't one of the three fixes identified;
    worth matching too if even closer fidelity is wanted.
  - `KERNEL_SIZE=25` is a much larger fraction of our 70-bar context (36%) than the
    article's 168-bar one (15%) -- kept for fidelity to the literal config value, not
    rescaled to be proportionally equivalent.
- **A genuinely channel-independent head previously regressed badly on this project**
  (the anchored-return-target notebook that tried one, early in this branch's history) --
  that was a different target (return decomposition, not RevIN raw-price), so it's not
  certain the same regression happens here, but watch OHLC RMSE/dir acc/coherence closely
  against `hf_patchtst_revin_no_volume` (fused head) for a sign of the same failure mode.
- **Two full backbones instead of one** -- roughly 2x the parameters and training compute
  of `hf_patchtst_revin_no_volume`.
- **One seed per setting.**
- **No output bounding at all** -- nothing stops a wild prediction if training is unstable.
- **No backtest yet** -- deferred until a target/volume/loss/architecture combination is
  picked.
- **`evaluate.py` does NOT support this checkpoint shape** -- this notebook computes its
  own metrics inline instead, same as every other notebook in this family.


## Commit + push results

Same pattern as `steven/colab_train.ipynb`'s own commit/push cells -- reused here so every
experiment notebook that runs on a fresh Colab VM handles git auth the same way.


In [ ]:
# Fresh Colab VM has no git identity configured -- needed for `commit` to work at all.
# Only sets it for this local clone (no --global), harmless to commit/share.
!git -C {REPO_DIR} config user.email "woodychang891121@gmail.com"
!git -C {REPO_DIR} config user.name "WoodyChang21"

commit_msg = f"chore(model): log {EXPERIMENT['name']} comparison_metrics from this Colab run"

!git add steven/comparison_metrics
!git status
!git commit -m {commit_msg!r}


In [ ]:
import getpass

# Fresh Colab VM has no stored GitHub credentials, so a plain `git push` over HTTPS
# can't authenticate. Prompting interactively (getpass masks it, and it's never written
# into this notebook's saved source/outputs) instead of hardcoding a token in a cell --
# a hardcoded token would get committed into git history the moment this notebook is
# pushed, which is a real credential leak. Needs a GitHub Personal Access Token with
# `repo` scope: https://github.com/settings/tokens
token = getpass.getpass("GitHub Personal Access Token: ")
push_url = f"https://{token}@github.com/WoodyChang21/ECE1508_GenAI.git"
!git -C {REPO_DIR} push {push_url} {BRANCH}
del token, push_url  # don't leave it sitting in a notebook-visible variable longer than needed
